# **Grouped Reinforcement Policy Optimization (GRPO)**

**Keywords:** Full fine tuning, RL, GRPO, `trl` library

[Original example](https://huggingface.co/learn/llm-course/en/chapter12/5) by [Maxime Labonne](https://https://huggingface.co/mlabonne).

**Environment:** Kaggle


## **Install dependencies**

In [ ]:
!pip install -qqq datasets==3.2.0 transformers==4.47.1 trl==0.14.0 peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.2 wandb==0.19.7 --progress-bar off

In [ ]:
!pip install -qqq -U bitsandbytes

In [ ]:
#!pip install -qqq wandb==0.19.7 --progress-bar off

`flash-attn`

**FlashAttention** is a fast and memory-efficient way to compute the attention mechanism used in Transformer models.

In [ ]:
#!pip install -qqq flash-attn --no-build-isolation --progress-bar off

**Note:** mey need to restart the kernel/session.

## **Imports**

In [1]:
import torch
# import wandb
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

2026-05-04 11:57:01.436921: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777895821.671975     143 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777895821.736568     143 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777895822.253971     143 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777895822.254019     143 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777895822.254023     143 computation_placer.cc:177] computation placer alr

## **Load Dataset**

In [ ]:
# Log to Weights & Biases
# wandb.login()

This [dataset](https://huggingface.co/datasets/mlabonne/smoltldr) was designed for the fine-tune of `HuggingFaceTB/SmolLM2-135M-Instruct` using **GRPO**. It is designed to summarize Reddit posts.

In [2]:
# Load dataset
dataset = load_dataset("mlabonne/smoltldr")
print(dataset)

README.md:   0%|          | 0.00/981 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.44M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 200
    })
})


## **Load Model**

In [3]:
# Load model
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
    #attn_implementation="flash_attention_2",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

In [4]:
# Load LoRA
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())

trainable params: 4,884,480 || all params: 139,399,488 || trainable%: 3.5039
None


## **Define Reward Function**

In [8]:
ideal_length = 50

def reward_len(completions, **kwargs):
    return [-abs(ideal_length - len(completion)) for completion in completions]

## **Define Training Arguments**

In [19]:
# Training arguments
training_args = GRPOConfig(
    output_dir="GRPO",
    learning_rate=2e-5,
    per_device_train_batch_size=2, # 8
    gradient_accumulation_steps=4, #2 
    max_prompt_length=512,
    max_completion_length=96,
    num_generations=8,
    optim="adamw_8bit",
    num_train_epochs=1,
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
    logging_steps=1,
)

In [20]:
# Trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_len,],
    args=training_args,
    train_dataset=dataset["train"],
)

In [21]:
# Train model
trainer.train()

Step,Training Loss
1,0.000000
2,0.000100
3,0.000100
4,0.000100
5,0.000100
6,0.000100
7,0.000100
8,0.000100
9,0.000100
10,0.000100


TrainOutput(global_step=250, training_loss=0.012677048802375794, metrics={'train_runtime': 10684.299, 'train_samples_per_second': 0.187, 'train_steps_per_second': 0.023, 'total_flos': 0.0, 'train_loss': 0.012677048802375794})

## **Push Model to Hub**

In [ ]:
# Save model
merged_model = trainer.model.merge_and_unload()

In [ ]:
# Push Model to Hub
# merged_model.push_to_hub("", private=False)

## **Generate Text**

In [ ]:
# Generate text
from transformers import pipeline

generator = pipeline("text-generation", model="/content/GRPO")

## Or use the model and tokenizer we defined earlier
# generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

generate_kwargs = {
    "max_new_tokens": 256,
    "do_sample": True,
    "temperature": 0.5,
    "min_p": 0.1,
}

generated_text = generator(messages, generate_kwargs=generate_kwargs)

print(generated_text)